# Q1. Top 10 suppliers and spend concentration

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** Which top 10 suppliers account for the largest share of Net PO value, and what share is
concentrated in the top 20% of suppliers?

Suppliers are grouped by supplier number, because several suppliers share a name (e.g. `Vendor`).

In [ ]:
sup = (w.groupby("supplier", dropna=False)["Net Order Value"].sum()
         .sort_values(ascending=False).reset_index(name="net"))
sup["share"] = sup["net"] / TOTAL
sup["cum_share"] = sup["share"].cumsum()
sup["ABC"] = pd.cut(sup["cum_share"], [0, .8, .95, 1.0001], labels=list("ABC"))

n20 = round(len(sup) * 0.20)
hhi = (sup["share"] ** 2).sum()
print(f"Suppliers: {len(sup)}")
print(f"Top 10 share: {sup.head(10)['share'].sum():.1%}")
print(f"Top 20% ({n20} suppliers) share: {sup.head(n20)['share'].sum():.1%}")
print(f"HHI: {hhi:.3f}  (equivalent to {1 / hhi:.0f} equal-sized suppliers)")
sup.head(10).style.format({"net": "{:,.0f}", "share": "{:.1%}", "cum_share": "{:.1%}"})

In [ ]:
top10 = sup.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(top10["supplier"], top10["net"], color=SERIES[0], height=0.65)
for y, (v, s) in enumerate(zip(top10["net"], top10["share"])):
    ax.text(v, y, f"  ${v / 1e3:,.0f}k · {s:.1%}", va="center", color=INK_2, fontsize=9)
ax.xaxis.set_major_formatter(usd)
ax.set_xlim(0, top10["net"].max() * 1.3)
ax.grid(axis="y", visible=False)
ax.set(title=f"Top 10 suppliers by Net PO value, US00, {PERIOD}", xlabel="Net PO value, USD")
save(fig, "q1_top10_suppliers")

In [ ]:
x = np.arange(1, len(sup) + 1) / len(sup)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, sup["cum_share"], color=SERIES[0], label="Cumulative share of Net PO value")
ax.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1, label="Equal spend per supplier")
for k, label in [(10, "Top 10"), (n20, "Top 20%")]:
    xv, yv = k / len(sup), sup["cum_share"].iat[k - 1]
    ax.plot(xv, yv, "o", color=SERIES[1], mec="#fcfcfb", mew=2)
    ax.annotate(f"{label}: {yv:.0%}", (xv, yv), xytext=(10, -14), textcoords="offset points", color=INK_2)
ax.axhline(0.8, color=MUTED, lw=1)
ax.text(1, 0.8, "80% (A class)", ha="right", va="bottom", color=INK_2, fontsize=9)
ax.xaxis.set_major_formatter(pct); ax.yaxis.set_major_formatter(pct)
ax.set(title="Spend concentration (Pareto curve)", xlabel="Share of suppliers (ranked by value)",
       ylabel="Cumulative share of Net PO value", xlim=(0, 1), ylim=(0, 1.02))
ax.legend(loc="lower right")
save(fig, "q1_pareto")

In [ ]:
(sup.groupby("ABC", observed=True)
    .agg(suppliers=("supplier", "size"), net=("net", "sum"))
    .assign(share_of_suppliers=lambda d: d.suppliers / d.suppliers.sum(),
            share_of_value=lambda d: d.net / d.net.sum())
    .style.format({"net": "{:,.0f}", "share_of_suppliers": "{:.0%}", "share_of_value": "{:.0%}"}))

## Findings for Antonio

**Finding.** 291 suppliers share $3.06M of Net PO value. The **top 10 hold 36.2%**, and the **top 20% (58 suppliers) hold 69.5%**. That is concentrated, but less than the classic 80/20 rule: 87 suppliers (30%) make up the A class (80% of value). The largest supplier, Szallito Angela, has 10.1%. HHI = 0.022, which is like having ~45 equal-sized suppliers, so no single supplier dominates.

**Decision it supports.** The ~60 A-class suppliers are where volume negotiations and long-term contracts pay off. The 134 C-class suppliers bring only 5% of value and are candidates for reducing the supplier base.

**Limitation / extra data.** Net PO value is ordered value, not invoiced spend or total cost of ownership. Many top suppliers are student test records (`szállító…`, `Bike supplier 000`), and the top supplier's value comes from one exercise item (spelt flour at $1,500/kg). Before Antonio relies on a supplier more, he needs quality (rejection rates), delivery reliability (goods-receipt vs planned dates) and lifecycle cost data.